# 🩺 MedSoru - Tıp Fakültesi Amfi Ses Kayıtları Transkripsiyon Sistemi
### Google Gemini 3.8 Flash (1M Token Çok Modlu Tıbbi Dil Modeli) ile Otomatik Çözümleme

Bu Google Colab defteri; paylaşılan Google Drive klasöründeki veya cihazınızdaki tıp dersi ses kayıtlarını (.m4a, .mp3, .wav):
1. **Tıbbi Terminoloji ve Müfredat Uyumu**: Latince anatomik terimler, farmakolojik etken maddeler, patoloji bulguları ve mikrobiyolojik etkenleri literatüre %100 uygun biçimde yazar.
2. **Ders & Kurul Tespiti**: Dosya adı ve ses içeriğinden kaydın hangi Kurula (Kurul 1-6), hangi branşa (Genetik, Üroloji, Patoloji, Farmakoloji, Halk Sağlığı vb.) ve hangi derse ait olduğunu otomatik tespit eder.
3. **Sınav İpuçları (High-Yield Pearls)**: Hocanın amfide 'burası sınavda çıkar', 'TUS'ta sorulur' dediği kısımları özel hap bilgi kutusu olarak çıkarır.
4. **Kota & Hız Limiti Yönetimi**: Gemini API hız limitlerini (RPM/TPM) aşmayacak şekilde güvenli aralıklarla çalışır ve ilerlemeyi kaydeder.

### 1. Adım: Gerekli Paketleri Yükleyin

In [ ]:
!pip install -q -U google-genai gdown

### 2. Adım: Google Drive'ı Bağlayın (Önerilen)
Eğer ses kayıtları Google Drive'ınızda ise aşağıdaki hücreyi çalıştırarak Drive'ınızı bağlayabilirsiniz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Adım: Ayarlar ve Gemini API Anahtarı

In [ ]:
import os
from google.colab import userdata

# @title ⚙️ Transkripsiyon Yapılandırması
# Gemini API Anahtarınızı Colab Secrets (Sol menüdeki anahtar simgesi) üzerinden veya doğrudan buraya girin:
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    GEMINI_API_KEY = "" # @param {type:"string"}

# Drive klasör yolu veya yerel ses dosyalarının yolu:
AUDIO_DIR = "/content/drive/MyDrive/Quick" # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/Meds_Transkriptler" # @param {type:"string"}
MODEL_NAME = "gemini-3.8-flash" # @param ["gemini-3.8-flash", "gemini-3.5-flash-lite"]
DELAY_BETWEEN_CALLS = 8 # @param {type:"slider", min:4, max:20, step:1}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Ayarlar kaydedildi. Ses Klasörü: {AUDIO_DIR}")
print(f"💾 Çıktı Klasörü: {OUTPUT_DIR}")

### 4. Adım (Alternatif): Drive Klasörünü Doğrudan İndirin
Eğer Drive'ı bağlamak yerine bağlantıdan (`1iwuuR8HFn4voHxGO2s_C9QCPuhfFq059`) doğrudan indirmek isterseniz:

In [ ]:
# @title 📥 Drive Klasörünü İndir (İsteğe bağlı)
DRIVE_FOLDER_ID = "1iwuuR8HFn4voHxGO2s_C9QCPuhfFq059" # @param {type:"string"}
!gdown --folder https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID} -O /content/ses_kayitlari
# AUDIO_DIR yolunu indirilen klasör yap:
if os.path.exists("/content/ses_kayitlari"):
    AUDIO_DIR = "/content/ses_kayitlari"
    print("✅ İndirilen ses klasörü aktif edildi:", AUDIO_DIR)

### 5. Adım: Tıbbi Transkripsiyon ve Ders Eşleştirme Motorunu Başlatın

In [ ]:
import time
import re
import json
from pathlib import Path
from google import genai
from google.genai import types

# Gemini İstemcisini Başlat
client = genai.Client(api_key=GEMINI_API_KEY)

# Tıbbi Sözlük ve Yazım Düzeltmeleri
TYPO_CORRECTIONS = {
    "üoloji": "üroloji", "uoloji": "üroloji",
    "saülığı": "sağlığı", "sauiligi": "sağlığı",
    "genetşk": "genetik", "genetsk": "genetik",
    "tbg": "Tıbbi Genetik", "hs": "Halk Sağlığı"
}

def clean_title(name):
    base = Path(name).stem.lower()
    for k, v in TYPO_CORRECTIONS.items():
        base = base.replace(k, v)
    return base

audio_path = Path(AUDIO_DIR)
supported_exts = {".m4a", ".mp3", ".wav", ".aac", ".ogg", ".flac"}
audio_files = [f for f in audio_path.glob("**/*") if f.suffix.lower() in supported_exts]
manifest_file = Path(OUTPUT_DIR) / "transcription_manifest.json"

manifest = {}
if manifest_file.exists():
    try:
        with open(manifest_file, "r", encoding="utf-8") as f:
            manifest = json.load(f)
    except:
        manifest = {}

print(f"🎯 Bulunan Toplam Ses Kaydı: {len(audio_files)}")

for idx, file in enumerate(audio_files, 1):
    file_key = file.stem
    if file_key in manifest and manifest[file_key].get("status") == "completed":
        print(f"⏩ [{idx}/{len(audio_files)}] Zaten Tamamlandı (Atlandı): {file.name}")
        continue

    print(f"\n🎙️ [{idx}/{len(audio_files)}] Yükleniyor: {file.name} ({file.stat().st_size / (1024*1024):.1f} MB)...")
    
    # 1. Gemini File API ile yükle
    up_file = client.files.upload(file=str(file), config={"display_name": file.name})
    
    # 2. İşlenme durumunu bekle
    while up_file.state == "PROCESSING":
        time.sleep(4)
        up_file = client.files.get(name=up_file.name)
        print(".", end="", flush=True)
        
    if up_file.state != "ACTIVE":
        print(f"\n❌ Dosya işlenemedi: {up_file.state}")
        continue

    print("\n🧠 Tıbbi Dil Motoru ile Transkribe Ediliyor...")
    cleaned_cand = clean_title(file.name)
    
    prompt = f"""Sen Tıp Fakültesi Dönem 3 amfi derslerini çözen Kıdemli Tıp Fakültesi Profesörüsün.
SES KAYDI: {file.name} (Aday İpucu: {cleaned_cand})

GÖREVLER:
1. GERÇEK DERSİ VE DİSİPLİNİ BUL: Sesin başındaki konuşmalardan ve işlenen tıbbi konulardan hareketle bu kaydın GERÇEKTE hangi Kurula (Kurul 1-6), hangi Tıp Branşına (örn. Tıbbi Genetik, Üroloji, Patoloji, Farmakoloji, vb.), hangi Ders Başlığına ve hangi Öğretim Üyesine ait olduğunu tespit et.
2. TIBBİ LİTERATÜR VE YAZIM: Bütün anatomik terimleri, hastalıkları, mikroorganizmaları ve ilaçları doğru tıbbi imlayla eksiksiz transkribe et.
3. SINAV İNCİLERİ: Hocanın özellikle sınav veya TUS için uyardığı yerleri 'High-Yield Pearls' olarak listele.

ÇIKTI FORMATI:
# 🩺 [Tespit Edilen Resmi Ders Adı]
> **Kurul:** [Kurul Bilgisi]  
> **Disiplin:** [Tıp Disiplini]  
> **Öğretim Üyesi:** [Hoca Adı]  
> **Kaynak Dosya:** `{file.name}`  

## 📌 Dersin Özeti ve Anahtar Kavramlar
[Özet]

## ⭐ Sınav & TUS Hap Bilgileri (High-Yield Pearls)
- [Kritik Bilgi 1]

## 📝 Ayrıntılı Ders Transkripti
### 1. [Bölüm Başlığı]
[Transkript metni...]
"""

    # 3. Model Üretimi ve Hata Yönetimi
    try:
        res = client.models.generate_content(
            model=MODEL_NAME,
            contents=[up_file, prompt],
            config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=8192)
        )
        md_text = res.text
        
        # Markdown dosyasını kaydet
        safe_out_name = re.sub(r'[^a-zA-Z0-9_\-\s]', '_', file.stem) + "_Transkript.md"
        out_file_path = Path(OUTPUT_DIR) / safe_out_name
        with open(out_file_path, "w", encoding="utf-8") as f:
            f.write(md_text)
            
        print(f"✅ Transkript Kaydedildi: {safe_out_name}")
        
        manifest[file_key] = {
            "filename": file.name,
            "output_file": safe_out_name,
            "status": "completed",
            "time": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        with open(manifest_file, "w", encoding="utf-8") as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)
            
    except Exception as e:
        print(f"❌ Transkripsiyon Hatası: {e}")
        
    finally:
        # Depo kotasını korumak için geçici dosyayı temizle
        try:
            client.files.delete(name=up_file.name)
        except:
            pass
            
    # Kota koruma beklemesi
    time.sleep(DELAY_BETWEEN_CALLS)

print("\n🎉 Bütün ses kayıtları başarıyla işlendi!")

### 6. Adım: Tüm Transkriptleri ZIP Olarak İndirin

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("/content/Meds_Transkriptler", 'zip', OUTPUT_DIR)
files.download("/content/Meds_Transkriptler.zip")
print("📦 ZIP paketi indiriliyor...")